In [2]:
from __future__ import annotations

import re
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import soundfile as sf

In [7]:
from collections import Counter

PROCESSED_DATASET_DIRECTORY = Path(
    "/Volumes/LaCie/DAIC-WOZ/Processed Dataset"
)

INPUT_AUDIO_SUFFIX = "_removed_interviewer.wav"

audio_files = sorted(
    path.resolve()
    for path in PROCESSED_DATASET_DIRECTORY.rglob(
        f"*{INPUT_AUDIO_SUFFIX}"
    )
    if path.is_file()
)

print(f"Total matching files: {len(audio_files)}")

participant_counts = Counter(
    path.parent.name
    for path in audio_files
)

print(
    "Unique immediate parent folders:",
    len(participant_counts),
)

print("\nFolders containing multiple matching audio files:")

for folder_name, count in sorted(
    participant_counts.items(),
    key=lambda item: (-item[1], item[0]),
):
    if count > 1:
        print(f"{folder_name}: {count}")
        for path in audio_files:
            if path.parent.name == folder_name:
                print(f"    {path}")

Total matching files: 295
Unique immediate parent folders: 185

Folders containing multiple matching audio files:
303_P: 2
    /Volumes/LaCie/DAIC-WOZ/Processed Dataset/303_P/._303_AUDIO_removed_interviewer.wav
    /Volumes/LaCie/DAIC-WOZ/Processed Dataset/303_P/303_AUDIO_removed_interviewer.wav
304_P: 2
    /Volumes/LaCie/DAIC-WOZ/Processed Dataset/304_P/._304_AUDIO_removed_interviewer.wav
    /Volumes/LaCie/DAIC-WOZ/Processed Dataset/304_P/304_AUDIO_removed_interviewer.wav
305_P: 2
    /Volumes/LaCie/DAIC-WOZ/Processed Dataset/305_P/._305_AUDIO_removed_interviewer.wav
    /Volumes/LaCie/DAIC-WOZ/Processed Dataset/305_P/305_AUDIO_removed_interviewer.wav
306_P: 2
    /Volumes/LaCie/DAIC-WOZ/Processed Dataset/306_P/._306_AUDIO_removed_interviewer.wav
    /Volumes/LaCie/DAIC-WOZ/Processed Dataset/306_P/306_AUDIO_removed_interviewer.wav
307_P: 2
    /Volumes/LaCie/DAIC-WOZ/Processed Dataset/307_P/._307_AUDIO_removed_interviewer.wav
    /Volumes/LaCie/DAIC-WOZ/Processed Dataset/307_P/307_A

In [8]:
appledouble_files = [
    path
    for path in audio_files
    if path.name.startswith("._")
]

real_audio_files = [
    path
    for path in audio_files
    if not path.name.startswith("._")
]

print(f"All matching paths: {len(audio_files)}")
print(f"AppleDouble metadata files: {len(appledouble_files)}")
print(f"Real audio files: {len(real_audio_files)}")

All matching paths: 295
AppleDouble metadata files: 110
Real audio files: 185


In [17]:
# ============================================================
# CONFIGURATION
# ============================================================

# Processed recordings will be placed here.
PROCESSED_DATASET_DIRECTORY = Path("/Volumes/LaCie/DAIC-WOZ/Processed Dataset")

EVENTS_CSV_PATH = (PROCESSED_DATASET_DIRECTORY / "acoustic_overlap_events.csv")

# Used only when audio_path in the CSV is missing or invalid.
AUDIO_ROOT_DIRECTORY = Path("/Volumes/LaCie/DAIC-WOZ/Dataset")

#PROCESSED_DATASET_DIRECTORY = Path("/Volumes/LaCie/DAIC-WOZ/Processed Dataset")

OUTPUT_SUMMARY_CSV = (PROCESSED_DATASET_DIRECTORY / "manual_interviewer_removal_summary.csv")

# Audio columns in acoustic_overlap_events.csv.
AUDIO_PATH_COLUMN = "audio_path"
AUDIO_FILENAME_COLUMN = "audio_filename"
PARTICIPANT_ID_COLUMN = "participant_id"

MANUAL_LABEL_COLUMN = "manual_label"
NOTES_COLUMN = "notes"

# A row qualifies when the label contains this text.
INTERVIEWER_LABEL_TEXT = "interviewer"

# Rows with exactly this label are ignored.
IGNORE_LABEL = "remove"

# Applied when no explicit "start" or "end" marker is present
# for that audio recording.
DEFAULT_BEGINNING_REMOVAL_SECONDS = 30.0
DEFAULT_END_REMOVAL_SECONDS = 30.0

# The files that should be processed.
INPUT_AUDIO_SUFFIX = "_removed_interviewer.wav"

# New output filename:
# example_removed_interviewer.wav
# becomes:
# example_removed_interviewer_manually_cleaned.wav
OUTPUT_FILENAME_SUFFIX = "_manually_cleaned"

# Prevent accidental overwriting.
OVERWRITE_EXISTING_OUTPUTS = True


# ============================================================
# TIMESTAMP PARSING
# ============================================================

def parse_timestamp(timestamp: str) -> float:
    """
    Convert a timestamp to seconds.

    Supported formats
    -----------------
    "15"          -> 15 seconds
    "15.5"        -> 15.5 seconds
    "9:15"        -> 9 minutes, 15 seconds
    "21:17.5"     -> 21 minutes, 17.5 seconds
    "1:02:15.5"   -> 1 hour, 2 minutes, 15.5 seconds
    "21.17.5"     -> 21 minutes, 17.5 seconds

    The last format is included because notes may contain values such
    as "21.17.5" instead of "21:17.5".
    """
    value = str(timestamp).strip()

    if not value:
        raise ValueError("Timestamp is empty.")

    # Standard colon-delimited formats.
    if ":" in value:
        parts = value.split(":")

        if len(parts) == 2:
            minutes = float(parts[0])
            seconds = float(parts[1])

            if seconds >= 60:
                raise ValueError(
                    f"Seconds must be less than 60 in timestamp {value!r}."
                )

            return minutes * 60 + seconds

        if len(parts) == 3:
            hours = float(parts[0])
            minutes = float(parts[1])
            seconds = float(parts[2])

            if minutes >= 60 or seconds >= 60:
                raise ValueError(
                    f"Invalid HH:MM:SS timestamp: {value!r}"
                )

            return hours * 3600 + minutes * 60 + seconds

        raise ValueError(
            f"Unsupported colon-delimited timestamp: {value!r}"
        )

    # Handle nonstandard MM.SS.fraction notation:
    # "21.17.5" -> 21 minutes + 17.5 seconds.
    dot_parts = value.split(".")

    if len(dot_parts) == 3 and all(
        part.isdigit() for part in dot_parts
    ):
        minutes = float(dot_parts[0])
        seconds = float(
            f"{dot_parts[1]}.{dot_parts[2]}"
        )

        if seconds >= 60:
            raise ValueError(
                f"Seconds must be less than 60 in timestamp {value!r}."
            )

        return minutes * 60 + seconds

    # Plain seconds, possibly with a decimal.
    try:
        seconds = float(value)
    except ValueError as error:
        raise ValueError(
            f"Could not parse timestamp: {value!r}"
        ) from error

    if seconds < 0:
        raise ValueError(
            f"Timestamp cannot be negative: {value!r}"
        )

    return seconds


def seconds_to_timestamp(seconds: float) -> str:
    """
    Convert seconds to HH:MM:SS.mmm.
    """
    seconds = max(0.0, float(seconds))

    hours = int(seconds // 3600)
    remaining = seconds % 3600

    minutes = int(remaining // 60)
    remaining %= 60

    whole_seconds = int(remaining)
    milliseconds = int(
        round((remaining - whole_seconds) * 1000)
    )

    if milliseconds == 1000:
        milliseconds = 0
        whole_seconds += 1

    if whole_seconds == 60:
        whole_seconds = 0
        minutes += 1

    if minutes == 60:
        minutes = 0
        hours += 1

    return (
        f"{hours:02d}:"
        f"{minutes:02d}:"
        f"{whole_seconds:02d}."
        f"{milliseconds:03d}"
    )


# ============================================================
# NOTE PARSING
# ============================================================

# Each endpoint may be:
# - start
# - end
# - seconds
# - MM:SS
# - HH:MM:SS
# - MM.SS.fraction
_TIMESTAMP_EXPRESSION = (
    r"(?:start|end|"
    r"\d+(?::\d+(?::\d+(?:\.\d+)?)?|(?:\.\d+){1,2})?)"
)

RANGE_PATTERN = re.compile(
    rf"(?P<left>{_TIMESTAMP_EXPRESSION})"
    rf"\s*-\s*"
    rf"(?P<right>{_TIMESTAMP_EXPRESSION})",
    flags=re.IGNORECASE,
)


def parse_notes(
    notes: str,
    audio_duration_seconds: float,
) -> tuple[list[tuple[float, float]], bool, bool]:
    """
    Parse all timestamp ranges from one notes field.

    Returns
    -------
    intervals
        Parsed removal intervals in seconds.

    has_explicit_start
        True when at least one range begins with "start".

    has_explicit_end
        True when at least one range ends with "end".
    """
    normalized_notes = str(notes).strip()

    # Normalize Unicode dashes copied from spreadsheets or documents.
    normalized_notes = (
        normalized_notes
        .replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
    )

    intervals: list[tuple[float, float]] = []
    has_explicit_start = False
    has_explicit_end = False

    matches = list(
        RANGE_PATTERN.finditer(normalized_notes)
    )

    if not matches:
        raise ValueError(
            f"No timestamp ranges were found in notes: "
            f"{normalized_notes!r}"
        )

    for match in matches:
        left_value = match.group("left").strip().lower()
        right_value = match.group("right").strip().lower()

        if left_value == "start":
            start_seconds = 0.0
            has_explicit_start = True
        elif left_value == "end":
            raise ValueError(
                "'end' cannot be used as the beginning of a range: "
                f"{match.group(0)!r}"
            )
        else:
            start_seconds = parse_timestamp(left_value)

        if right_value == "end":
            stop_seconds = audio_duration_seconds
            has_explicit_end = True
        elif right_value == "start":
            raise ValueError(
                "'start' cannot be used as the end of a range: "
                f"{match.group(0)!r}"
            )
        else:
            stop_seconds = parse_timestamp(right_value)

        # Clip intervals to the actual recording.
        start_seconds = max(
            0.0,
            min(start_seconds, audio_duration_seconds),
        )
        stop_seconds = max(
            0.0,
            min(stop_seconds, audio_duration_seconds),
        )

        if stop_seconds <= start_seconds:
            raise ValueError(
                f"Range stop must occur after range start: "
                f"{match.group(0)!r}"
            )

        intervals.append(
            (start_seconds, stop_seconds)
        )

    return (
        intervals,
        has_explicit_start,
        has_explicit_end,
    )


# ============================================================
# INTERVAL UTILITIES
# ============================================================

def merge_intervals(
    intervals: Iterable[tuple[float, float]],
) -> list[tuple[float, float]]:
    """
    Merge overlapping or directly adjacent intervals.
    """
    sorted_intervals = sorted(
        (
            float(start),
            float(stop),
        )
        for start, stop in intervals
        if stop > start
    )

    if not sorted_intervals:
        return []

    merged: list[list[float]] = [
        [
            sorted_intervals[0][0],
            sorted_intervals[0][1],
        ]
    ]

    for current_start, current_stop in sorted_intervals[1:]:
        previous_start, previous_stop = merged[-1]

        if current_start <= previous_stop:
            merged[-1][1] = max(
                previous_stop,
                current_stop,
            )
        else:
            merged.append(
                [current_start, current_stop]
            )

    return [
        (start, stop)
        for start, stop in merged
    ]


def invert_intervals(
    removal_intervals: list[tuple[float, float]],
    audio_duration_seconds: float,
) -> list[tuple[float, float]]:
    """
    Return the intervals that should be retained.
    """
    keep_intervals: list[tuple[float, float]] = []

    current_position = 0.0

    for remove_start, remove_stop in removal_intervals:
        if remove_start > current_position:
            keep_intervals.append(
                (current_position, remove_start)
            )

        current_position = max(
            current_position,
            remove_stop,
        )

    if current_position < audio_duration_seconds:
        keep_intervals.append(
            (current_position, audio_duration_seconds)
        )

    return keep_intervals


def format_intervals(
    intervals: Iterable[tuple[float, float]],
) -> str:
    """
    Format intervals for the summary CSV.
    """
    return "; ".join(
        f"{seconds_to_timestamp(start)}-"
        f"{seconds_to_timestamp(stop)}"
        for start, stop in intervals
    )


# ============================================================
# AUDIO PATH RESOLUTION
# ============================================================

def find_all_processed_audio_files() -> list[Path]:
    """
    Find real interviewer-removed WAV files in participant folders.

    Excludes:
    - macOS AppleDouble files beginning with "._"
    - other hidden files
    - already manually cleaned output files
    """
    if not PROCESSED_DATASET_DIRECTORY.is_dir():
        raise FileNotFoundError(
            "Processed dataset directory does not exist: "
            f"{PROCESSED_DATASET_DIRECTORY}"
        )

    audio_files: list[Path] = []

    for audio_path in PROCESSED_DATASET_DIRECTORY.rglob(
        f"*{INPUT_AUDIO_SUFFIX}"
    ):
        if not audio_path.is_file():
            continue

        # Ignore macOS AppleDouble metadata files.
        if audio_path.name.startswith("._"):
            continue

        # Ignore any other hidden file.
        if audio_path.name.startswith("."):
            continue

        # Extra safeguard against processing cleaned outputs.
        if OUTPUT_FILENAME_SUFFIX in audio_path.stem:
            continue

        audio_files.append(audio_path.resolve())

    return sorted(audio_files)

def resolve_audio_path(
    group: pd.DataFrame,
) -> Path:
    """
    Resolve an audio path for one grouped recording.

    Priority
    --------
    1. Existing audio_path value in the CSV.
    2. Search AUDIO_ROOT_DIRECTORY recursively by audio_filename.
    """
    if AUDIO_PATH_COLUMN in group.columns:
        for path_value in group[AUDIO_PATH_COLUMN].dropna():
            candidate = Path(str(path_value)).expanduser()

            if candidate.is_file():
                return candidate.resolve()

    if AUDIO_FILENAME_COLUMN not in group.columns:
        raise KeyError(
            f"The CSV must contain either "
            f"{AUDIO_PATH_COLUMN!r} or "
            f"{AUDIO_FILENAME_COLUMN!r}."
        )

    filenames = (
        group[AUDIO_FILENAME_COLUMN]
        .dropna()
        .astype(str)
        .str.strip()
    )

    if filenames.empty:
        raise ValueError(
            "The grouped rows do not contain an audio filename."
        )

    audio_filename = filenames.iloc[0]

    matches = list(
        AUDIO_ROOT_DIRECTORY.rglob(audio_filename)
    )

    if not matches:
        raise FileNotFoundError(
            f"Could not find {audio_filename!r} under "
            f"{AUDIO_ROOT_DIRECTORY}."
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple audio files named {audio_filename!r} "
            f"were found: {matches}"
        )

    return matches[0].resolve()


# ============================================================
# AUDIO EDITING
# ============================================================

def remove_audio_intervals(
    input_audio_path: Path,
    output_audio_path: Path,
    removal_intervals: list[tuple[float, float]],
) -> tuple[float, float, int, str]:
    """
    Remove intervals from a WAV file and save the retained audio.

    Returns
    -------
    original_duration_seconds
    output_duration_seconds
    sample_rate
    original_subtype
    """
    try:
        audio_data, sample_rate = sf.read(
            str(input_audio_path),
            always_2d=True,
        )
        audio_info = sf.info(
            str(input_audio_path)
        )
    except Exception as error:
        raise RuntimeError(
            f"Could not read WAV file: {input_audio_path}"
        ) from error

    if sample_rate <= 0:
        raise ValueError(
            f"Invalid sample rate for {input_audio_path}."
        )

    total_frames = audio_data.shape[0]
    original_duration = total_frames / sample_rate

    keep_intervals = invert_intervals(
        removal_intervals=removal_intervals,
        audio_duration_seconds=original_duration,
    )

    retained_chunks: list[np.ndarray] = []

    for keep_start, keep_stop in keep_intervals:
        start_frame = int(
            round(keep_start * sample_rate)
        )
        stop_frame = int(
            round(keep_stop * sample_rate)
        )

        start_frame = max(
            0,
            min(start_frame, total_frames),
        )
        stop_frame = max(
            0,
            min(stop_frame, total_frames),
        )

        if stop_frame > start_frame:
            retained_chunks.append(
                audio_data[start_frame:stop_frame]
            )

    if not retained_chunks:
        raise ValueError(
            "The removal intervals would delete the entire "
            f"recording: {input_audio_path.name}"
        )

    cleaned_audio = np.concatenate(
        retained_chunks,
        axis=0,
    )

    output_audio_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if output_audio_path.exists() and not OVERWRITE_EXISTING_OUTPUTS:
        raise FileExistsError(
            f"Output already exists: {output_audio_path}"
        )

    # Preserve the original WAV subtype when possible.
    output_subtype = audio_info.subtype

    try:
        sf.write(
            str(output_audio_path),
            cleaned_audio,
            sample_rate,
            subtype=output_subtype,
        )
    except Exception as error:
        raise RuntimeError(
            f"Could not write cleaned WAV file: "
            f"{output_audio_path}"
        ) from error

    output_duration = (
        cleaned_audio.shape[0] / sample_rate
    )

    return (
        float(original_duration),
        float(output_duration),
        int(sample_rate),
        str(output_subtype),
    )


# ============================================================
# DATA PROCESSING
# ============================================================

def normalize_audio_identifier(value: object) -> str:
    """
    Normalize an audio path or filename for matching.

    Matching is based on the filename, case-insensitively.
    """
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if not text:
        return ""

    return Path(text).name.casefold()

def build_annotation_lookup(
    events_df: pd.DataFrame,
) -> tuple[dict[str, pd.DataFrame], set[str]]:
    """
    Organize annotations by audio filename.

    Returns
    -------
    interviewer_groups
        Maps each audio filename to qualifying interviewer rows
        containing nonempty notes.

    recordings_marked_for_removal
        Audio filenames for which at least one row has the exact
        manual label "remove".
    """
    events_df = events_df.copy()

    labels = (
        events_df[MANUAL_LABEL_COLUMN]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    notes = (
        events_df[NOTES_COLUMN]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    if AUDIO_FILENAME_COLUMN in events_df.columns:
        identifiers = events_df[AUDIO_FILENAME_COLUMN].apply(
            normalize_audio_identifier
        )
    elif AUDIO_PATH_COLUMN in events_df.columns:
        identifiers = events_df[AUDIO_PATH_COLUMN].apply(
            normalize_audio_identifier
        )
    else:
        raise KeyError(
            f"The CSV must contain {AUDIO_FILENAME_COLUMN!r} "
            f"or {AUDIO_PATH_COLUMN!r}."
        )

    events_df["_normalized_audio_identifier"] = identifiers

    remove_mask = labels.str.casefold().eq(
        IGNORE_LABEL.casefold()
    )

    recordings_marked_for_removal = set(
        events_df.loc[
            remove_mask,
            "_normalized_audio_identifier",
        ]
    )

    recordings_marked_for_removal.discard("")

    interviewer_mask = labels.str.contains(
        INTERVIEWER_LABEL_TEXT,
        case=False,
        regex=False,
        na=False,
    )

    qualifying_mask = (
        interviewer_mask
        & notes.ne("")
        & ~remove_mask
        & events_df["_normalized_audio_identifier"].ne("")
    )

    qualifying_df = events_df.loc[
        qualifying_mask
    ].copy()

    interviewer_groups = {
        audio_identifier: group.copy()
        for audio_identifier, group in qualifying_df.groupby(
            "_normalized_audio_identifier",
            sort=False,
        )
    }

    return (
        interviewer_groups,
        recordings_marked_for_removal,
    )

def process_discovered_audio_file(
    audio_path: Path,
    annotation_group: pd.DataFrame | None,
) -> dict[str, object]:
    """
    Process one audio recording.

    If annotation_group is None or empty, only the default first
    and last 30-second removals are applied.
    """
    try:
        audio_info = sf.info(str(audio_path))
    except Exception as error:
        raise RuntimeError(
            f"Could not read audio metadata: {audio_path}"
        ) from error

    if audio_info.samplerate <= 0:
        raise ValueError(
            f"Invalid sample rate for {audio_path}."
        )

    audio_duration = (
        audio_info.frames / audio_info.samplerate
    )

    all_intervals: list[tuple[float, float]] = []

    has_any_explicit_start = False
    has_any_explicit_end = False

    parsed_notes: list[str] = []
    annotation_row_count = 0

    if annotation_group is not None and not annotation_group.empty:
        annotation_row_count = len(annotation_group)

        for row_index, row in annotation_group.iterrows():
            notes = str(row[NOTES_COLUMN]).strip()

            try:
                (
                    row_intervals,
                    row_has_start,
                    row_has_end,
                ) = parse_notes(
                    notes=notes,
                    audio_duration_seconds=audio_duration,
                )
            except Exception as error:
                raise ValueError(
                    f"Could not parse notes in CSV row "
                    f"{row_index}: {notes!r}. {error}"
                ) from error

            all_intervals.extend(row_intervals)

            has_any_explicit_start = (
                has_any_explicit_start
                or row_has_start
            )

            has_any_explicit_end = (
                has_any_explicit_end
                or row_has_end
            )

            parsed_notes.append(notes)

    # Add the first 30 seconds unless "start" was explicitly used
    # somewhere in this recording's annotations.
    if not has_any_explicit_start:
        beginning_stop = min(
            DEFAULT_BEGINNING_REMOVAL_SECONDS,
            audio_duration,
        )

        if beginning_stop > 0:
            all_intervals.append(
                (0.0, beginning_stop)
            )

    # Add the last 30 seconds unless "end" was explicitly used
    # somewhere in this recording's annotations.
    if not has_any_explicit_end:
        ending_start = max(
            0.0,
            audio_duration
            - DEFAULT_END_REMOVAL_SECONDS,
        )

        if ending_start < audio_duration:
            all_intervals.append(
                (ending_start, audio_duration)
            )

    merged_removal_intervals = merge_intervals(
        all_intervals
    )

    output_filename = (
        f"{audio_path.stem}"
        f"{OUTPUT_FILENAME_SUFFIX}"
        f"{audio_path.suffix}"
    )

    output_audio_path = (
        audio_path.parent
        / output_filename
    )

    (
        original_duration,
        output_duration,
        sample_rate,
        subtype,
    ) = remove_audio_intervals(
        input_audio_path=audio_path,
        output_audio_path=output_audio_path,
        removal_intervals=merged_removal_intervals,
    )

    removed_duration = (
        original_duration - output_duration
    )

    participant_id = audio_path.parent.name

    return {
        "participant_id": participant_id,
        "audio_filename": audio_path.name,
        "input_audio_path": str(audio_path),
        "output_audio_path": str(output_audio_path),
        "had_interviewer_annotations": (
            annotation_row_count > 0
        ),
        "annotation_row_count": annotation_row_count,
        "original_notes": " | ".join(parsed_notes),
        "explicit_start_present": (
            has_any_explicit_start
        ),
        "explicit_end_present": (
            has_any_explicit_end
        ),
        "default_first_30_seconds_removed": (
            not has_any_explicit_start
        ),
        "default_last_30_seconds_removed": (
            not has_any_explicit_end
        ),
        "removal_intervals": format_intervals(
            merged_removal_intervals
        ),
        "number_of_removal_intervals": len(
            merged_removal_intervals
        ),
        "original_duration_seconds": round(
            original_duration,
            3,
        ),
        "removed_duration_seconds": round(
            removed_duration,
            3,
        ),
        "output_duration_seconds": round(
            output_duration,
            3,
        ),
        "sample_rate": sample_rate,
        "audio_subtype": subtype,
        "status": "success",
        "error": "",
    }

def validate_columns(dataframe: pd.DataFrame) -> None:
    """
    Validate required annotation columns.
    """
    required_columns = {
        MANUAL_LABEL_COLUMN,
        NOTES_COLUMN,
    }

    missing_columns = required_columns.difference(
        dataframe.columns
    )

    if missing_columns:
        raise KeyError(
            "The events CSV is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    if (
        AUDIO_PATH_COLUMN not in dataframe.columns
        and AUDIO_FILENAME_COLUMN not in dataframe.columns
    ):
        raise KeyError(
            f"The events CSV must contain "
            f"{AUDIO_PATH_COLUMN!r} or "
            f"{AUDIO_FILENAME_COLUMN!r}."
        )


def filter_qualifying_rows(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Keep rows with interviewer labels and nonempty notes.

    Rows labeled exactly "remove" are excluded.
    """
    labels = (
        dataframe[MANUAL_LABEL_COLUMN]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    notes = (
        dataframe[NOTES_COLUMN]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    interviewer_mask = labels.str.contains(
        INTERVIEWER_LABEL_TEXT,
        case=False,
        regex=False,
        na=False,
    )

    ignore_mask = labels.str.casefold().eq(
        IGNORE_LABEL.casefold()
    )

    notes_present_mask = notes.ne("")

    return dataframe.loc[
        interviewer_mask
        & ~ignore_mask
        & notes_present_mask
    ].copy()


def create_grouping_key(
    dataframe: pd.DataFrame,
) -> pd.Series:
    """
    Create a stable per-recording grouping key.
    """
    if AUDIO_PATH_COLUMN in dataframe.columns:
        audio_paths = (
            dataframe[AUDIO_PATH_COLUMN]
            .fillna("")
            .astype(str)
            .str.strip()
        )
    else:
        audio_paths = pd.Series(
            "",
            index=dataframe.index,
            dtype="string",
        )

    if AUDIO_FILENAME_COLUMN in dataframe.columns:
        audio_filenames = (
            dataframe[AUDIO_FILENAME_COLUMN]
            .fillna("")
            .astype(str)
            .str.strip()
        )
    else:
        audio_filenames = pd.Series(
            "",
            index=dataframe.index,
            dtype="string",
        )

    # Prefer the path, but fall back to the filename.
    return audio_paths.where(
        audio_paths.ne(""),
        audio_filenames,
    )


def process_audio_group(
    group: pd.DataFrame,
) -> dict[str, object]:
    """
    Process all annotation rows belonging to one audio recording.
    """
    audio_path = resolve_audio_path(group)

    audio_info = sf.info(str(audio_path))
    audio_duration = (
        audio_info.frames
        / audio_info.samplerate
    )

    all_intervals: list[tuple[float, float]] = []
    has_any_explicit_start = False
    has_any_explicit_end = False
    parsed_notes: list[str] = []

    for row_index, row in group.iterrows():
        notes = str(row[NOTES_COLUMN]).strip()

        try:
            (
                row_intervals,
                row_has_start,
                row_has_end,
            ) = parse_notes(
                notes=notes,
                audio_duration_seconds=audio_duration,
            )
        except Exception as error:
            raise ValueError(
                f"Could not parse notes in CSV row "
                f"{row_index}: {notes!r}. {error}"
            ) from error

        all_intervals.extend(row_intervals)

        has_any_explicit_start = (
            has_any_explicit_start
            or row_has_start
        )

        has_any_explicit_end = (
            has_any_explicit_end
            or row_has_end
        )

        parsed_notes.append(notes)

    # Add the default first 30-second removal only when no
    # qualifying annotation uses "start".
    if not has_any_explicit_start:
        beginning_stop = min(
            DEFAULT_BEGINNING_REMOVAL_SECONDS,
            audio_duration,
        )

        if beginning_stop > 0:
            all_intervals.append(
                (0.0, beginning_stop)
            )

    # Add the default final 30-second removal only when no
    # qualifying annotation uses "end".
    if not has_any_explicit_end:
        ending_start = max(
            0.0,
            audio_duration
            - DEFAULT_END_REMOVAL_SECONDS,
        )

        if ending_start < audio_duration:
            all_intervals.append(
                (ending_start, audio_duration)
            )

    merged_removal_intervals = merge_intervals(
        all_intervals
    )

    output_filename = (
        f"{audio_path.stem}"
        f"{OUTPUT_FILENAME_SUFFIX}"
        f"{audio_path.suffix}"
    )
    
    output_audio_path = (
        audio_path.parent
        / output_filename
    )

    (
        original_duration,
        output_duration,
        sample_rate,
        subtype,
    ) = remove_audio_intervals(
        input_audio_path=audio_path,
        output_audio_path=output_audio_path,
        removal_intervals=merged_removal_intervals,
    )

    removed_duration = (
        original_duration - output_duration
    )

    participant_id = ""

    if PARTICIPANT_ID_COLUMN in group.columns:
        participant_values = (
            group[PARTICIPANT_ID_COLUMN]
            .dropna()
            .astype(str)
            .str.strip()
        )

        if not participant_values.empty:
            participant_id = participant_values.iloc[0]

    return {
        "participant_id": participant_id,
        "audio_filename": audio_path.name,
        "input_audio_path": str(audio_path),
        "output_audio_path": str(output_audio_path),
        "annotation_row_count": len(group),
        "original_notes": " | ".join(parsed_notes),
        "explicit_start_present": has_any_explicit_start,
        "explicit_end_present": has_any_explicit_end,
        "default_first_30_seconds_removed": (
            not has_any_explicit_start
        ),
        "default_last_30_seconds_removed": (
            not has_any_explicit_end
        ),
        "removal_intervals": format_intervals(
            merged_removal_intervals
        ),
        "number_of_removal_intervals": len(
            merged_removal_intervals
        ),
        "original_duration_seconds": round(
            original_duration,
            3,
        ),
        "removed_duration_seconds": round(
            removed_duration,
            3,
        ),
        "output_duration_seconds": round(
            output_duration,
            3,
        ),
        "sample_rate": sample_rate,
        "audio_subtype": subtype,
        "status": "success",
        "error": "",
    }


def process_all_processed_audio() -> pd.DataFrame:
    """
    Process every interviewer-removed audio file in the processed
    dataset.

    For files with qualifying manual annotations:
        - remove annotated intervals
        - add the default beginning/end removals when "start" or
          "end" is absent

    For files without qualifying manual annotations:
        - remove the first 30 seconds
        - remove the last 30 seconds

    Recordings labeled exactly "remove" are skipped.
    """
    if not EVENTS_CSV_PATH.is_file():
        raise FileNotFoundError(
            f"Events CSV does not exist: "
            f"{EVENTS_CSV_PATH}"
        )

    events_df = pd.read_csv(
        EVENTS_CSV_PATH,
        dtype="string",
    )

    validate_columns(events_df)

    (
        annotation_lookup,
        recordings_marked_for_removal,
    ) = build_annotation_lookup(events_df)

    audio_files = find_all_processed_audio_files()

    print(
        f"Processed audio files found: {len(audio_files)}"
    )
    print(
        "Files with qualifying interviewer annotations: "
        f"{len(annotation_lookup)}"
    )
    print(
        "Files marked for manual removal: "
        f"{len(recordings_marked_for_removal)}"
    )

    if not audio_files:
        print(
            "No processed audio files were found."
        )
        return pd.DataFrame()

    summary_rows: list[dict[str, object]] = []

    for recording_number, audio_path in enumerate(
        audio_files,
        start=1,
    ):
        audio_identifier = normalize_audio_identifier(
            audio_path.name
        )

        print(
            f"[{recording_number}/{len(audio_files)}] "
            f"Processing {audio_path.name}"
        )

        # Skip any recording where at least one CSV row was
        # manually labeled exactly "remove".
        if audio_identifier in recordings_marked_for_removal:
            print(
                "  Skipped because the recording is marked "
                "'remove'."
            )

            summary_rows.append(
                {
                    "participant_id": audio_path.parent.name,
                    "audio_filename": audio_path.name,
                    "input_audio_path": str(audio_path),
                    "output_audio_path": "",
                    "had_interviewer_annotations": (
                        audio_identifier in annotation_lookup
                    ),
                    "annotation_row_count": 0,
                    "original_notes": "",
                    "explicit_start_present": pd.NA,
                    "explicit_end_present": pd.NA,
                    "default_first_30_seconds_removed": False,
                    "default_last_30_seconds_removed": False,
                    "removal_intervals": "",
                    "number_of_removal_intervals": 0,
                    "original_duration_seconds": pd.NA,
                    "removed_duration_seconds": pd.NA,
                    "output_duration_seconds": pd.NA,
                    "sample_rate": pd.NA,
                    "audio_subtype": "",
                    "status": "skipped_manual_remove",
                    "error": "",
                }
            )

            pd.DataFrame(summary_rows).to_csv(
                OUTPUT_SUMMARY_CSV,
                index=False,
            )

            continue

        annotation_group = annotation_lookup.get(
            audio_identifier
        )

        try:
            summary_row = process_discovered_audio_file(
                audio_path=audio_path,
                annotation_group=annotation_group,
            )

            summary_rows.append(summary_row)

            if annotation_group is None:
                print(
                    "  No interviewer annotations; removed "
                    "default first and last 30 seconds."
                )
            else:
                print(
                    "  Applied interviewer annotations and "
                    "applicable boundary defaults."
                )

            print(
                "  Removed intervals: "
                f"{summary_row['removal_intervals']}"
            )

            print(
                "  Output: "
                f"{summary_row['output_audio_path']}"
            )

        except FileExistsError as error:
            print(
                f"  Skipped existing output: {error}"
            )

            summary_rows.append(
                {
                    "participant_id": audio_path.parent.name,
                    "audio_filename": audio_path.name,
                    "input_audio_path": str(audio_path),
                    "output_audio_path": str(
                        audio_path.parent
                        / (
                            f"{audio_path.stem}"
                            f"{OUTPUT_FILENAME_SUFFIX}"
                            f"{audio_path.suffix}"
                        )
                    ),
                    "had_interviewer_annotations": (
                        annotation_group is not None
                    ),
                    "annotation_row_count": (
                        len(annotation_group)
                        if annotation_group is not None
                        else 0
                    ),
                    "original_notes": "",
                    "explicit_start_present": pd.NA,
                    "explicit_end_present": pd.NA,
                    "default_first_30_seconds_removed": pd.NA,
                    "default_last_30_seconds_removed": pd.NA,
                    "removal_intervals": "",
                    "number_of_removal_intervals": pd.NA,
                    "original_duration_seconds": pd.NA,
                    "removed_duration_seconds": pd.NA,
                    "output_duration_seconds": pd.NA,
                    "sample_rate": pd.NA,
                    "audio_subtype": "",
                    "status": "skipped_existing_output",
                    "error": str(error),
                }
            )

        except Exception as error:
            error_message = (
                f"{type(error).__name__}: {error}"
            )

            print(
                f"  Failed: {error_message}"
            )

            summary_rows.append(
                {
                    "participant_id": audio_path.parent.name,
                    "audio_filename": audio_path.name,
                    "input_audio_path": str(audio_path),
                    "output_audio_path": "",
                    "had_interviewer_annotations": (
                        annotation_group is not None
                    ),
                    "annotation_row_count": (
                        len(annotation_group)
                        if annotation_group is not None
                        else 0
                    ),
                    "original_notes": "",
                    "explicit_start_present": pd.NA,
                    "explicit_end_present": pd.NA,
                    "default_first_30_seconds_removed": pd.NA,
                    "default_last_30_seconds_removed": pd.NA,
                    "removal_intervals": "",
                    "number_of_removal_intervals": pd.NA,
                    "original_duration_seconds": pd.NA,
                    "removed_duration_seconds": pd.NA,
                    "output_duration_seconds": pd.NA,
                    "sample_rate": pd.NA,
                    "audio_subtype": "",
                    "status": "failed",
                    "error": error_message,
                }
            )

        # Save progress after each recording.
        pd.DataFrame(summary_rows).to_csv(
            OUTPUT_SUMMARY_CSV,
            index=False,
        )

    summary_df = pd.DataFrame(summary_rows)

    print("\nProcessing complete.")

    successful_count = (
        summary_df["status"].eq("success").sum()
    )

    manual_remove_count = (
        summary_df["status"]
        .eq("skipped_manual_remove")
        .sum()
    )

    existing_output_count = (
        summary_df["status"]
        .eq("skipped_existing_output")
        .sum()
    )

    failed_count = (
        summary_df["status"].eq("failed").sum()
    )

    print(
        f"Successful recordings: {successful_count}"
    )
    print(
        f"Skipped manual removals: {manual_remove_count}"
    )
    print(
        "Skipped existing outputs: "
        f"{existing_output_count}"
    )
    print(
        f"Failed recordings: {failed_count}"
    )

    print(
        f"Summary CSV: {OUTPUT_SUMMARY_CSV}"
    )

    return summary_df

In [18]:
# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    summary_df = process_all_processed_audio()

Processed audio files found: 186
Files with qualifying interviewer annotations: 64
Files marked for manual removal: 5
[1/186] Processing 300_AUDIO_removed_interviewer.wav
  No interviewer annotations; removed default first and last 30 seconds.
  Removed intervals: 00:00:00.000-00:00:30.000; 00:06:15.122-00:06:45.122
  Output: /Volumes/LaCie/DAIC-WOZ/Processed Dataset/300_P/300_AUDIO_removed_interviewer_manually_cleaned.wav
[2/186] Processing 301_AUDIO_removed_interviewer.wav
  Applied interviewer annotations and applicable boundary defaults.
  Removed intervals: 00:00:00.000-00:00:30.000; 00:10:29.000-00:10:39.882
  Output: /Volumes/LaCie/DAIC-WOZ/Processed Dataset/301_P/301_AUDIO_removed_interviewer_manually_cleaned.wav
[3/186] Processing 302_AUDIO_removed_interviewer.wav
  No interviewer annotations; removed default first and last 30 seconds.
  Removed intervals: 00:00:00.000-00:00:30.000; 00:08:15.770-00:08:45.770
  Output: /Volumes/LaCie/DAIC-WOZ/Processed Dataset/302_P/302_AUDIO_r